<a href="https://colab.research.google.com/github/kimyejean02/energy_efficient_llm_inference/blob/main/Energy_Efficient_LLM_Inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Installs and check nvidia-smi is working

!pip install -q transformers torch accelerate pynvml datasets pandas
!nvidia-smi

Mon Oct 19 20:50:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
# TODO: Load dataset with prompts
# Example: WildChat dataset

!pip install -q datasets pandas

from datasets import load_dataset
import pandas as pd

ds = load_dataset(
    "allenai/WildChat",
    split="train",
    streaming=True
)

rows = []
target = 3000

for row in ds:
    if row.get("language") != "English":
        continue

    conversation = row.get("conversation", [])
    prompt = next(
        (
            turn.get("content", "").strip()
            for turn in conversation
            if turn.get("role") == "user"
            and turn.get("content", "").strip()
        ),
        ""
    )

    if not prompt or len(prompt) < 10:
        continue

    rows.append({
        "conversation_id": row.get("conversation_id", ""),
        "prompt": prompt
    })

    if len(rows) >= target:
        break

df = pd.DataFrame(rows)
df.to_csv("wildchat_sample.csv", index=False)

print(f"Collected {len(df)} prompts")
display(df.head(10))

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

Collected 3000 prompts


,conversation_id,prompt
0,26c5dc109920789f9199ff9b37acb8c1,"Write a very long, elaborate, descriptive and ..."
1,e87a1aeb9aafa35c00da39ddeb1139a0,what are you?
2,c3415a9e401ff379f29fe3ce02e500dc,Write an engaging and a constructive article f...
3,ec6578cd9d69130a769ea307b6e7a874,CONSTRAINTS:\n\n1. ~4000 word limit for short ...
4,c9fdedb0e222c9251e5fab2b0784240c,Write an engaging and a constructive article f...
5,01e1f36d3f0cf0c4acb8e15bc7f4ed10,CONSTRAINTS:\n\n1. ~4000 word limit for short ...
6,bf21a7138ef8028bee411c1f271ef94c,"Write a very long, elaborate, descriptive and ..."
7,a0d800583ab19bb20299af3925f9d6cf,CONSTRAINTS:\n\n1. ~4000 word limit for short ...
8,cc9c65c7200e499761e9b8bf6979f904,"Chris is quite innocent, shy, cheerful, and ha..."
9,0bdd8d0421b41716c666fc6450302b9e,Write an engaging and a constructive article f...


In [ ]:
# TODO: Classify prompts into categories
# Example: Zero-shot-classification

!pip install -q transformers torch

from transformers import pipeline

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0  # use GPU in Colab
)

labels = [
    "practical guidance and advice",
    "seeking factual information or explanations",
    "writing, editing, rewriting, or summarization"
]

def classify_prompt(prompt):
    # TODO

In [ ]:
# TODO: Data cleaning to make sure prompts are valid?

In [ ]:
# TODO: Save prompts into a folder so they don't need to be regenerated every time

from google.colab import drive
drive.mount('/content/drive')

import os
import shutil

# Create a folder
folder = "/content/drive/MyDrive/EEC_LLM_Project"
os.makedirs(folder, exist_ok=True)

# Save generated files
for filename in [
    "wildchat_sample.csv"
]:
    if os.path.exists(filename):
        shutil.copy2(
            filename,
            os.path.join(folder, filename)
        )
        print(f"Saved {filename} to Google Drive")

In [ ]:
# Sample GPU Baseline Power

!pip install -q nvidia-ml-py

import pynvml

pynvml.nvmlInit()
gpu = pynvml.nvmlDeviceGetHandleByIndex(0)

print("GPU:", pynvml.nvmlDeviceGetName(gpu))
print("Power:", pynvml.nvmlDeviceGetPowerUsage(gpu) / 1000, "W")
print("Memory:", pynvml.nvmlDeviceGetMemoryInfo(gpu).total / 1024**3, "GB")

GPU: NVIDIA A100-SXM4-40GB
Power: 44.485 W
Memory: 40.0 GB


In [ ]:
# Download models and perform inference with power sampling

import time
import threading
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM


MODELS = {
    "small_1.5B": "Qwen/Qwen2.5-1.5B-Instruct",
    "large_3B": "Qwen/Qwen2.5-3B-Instruct",
}

QUERIES = [
    # TODO: Fill with prompts from dataset
]

MAX_NEW_TOKENS = 128
POWER_SAMPLE_INTERVAL = 0.05  # seconds


def synchronize_gpu():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

# Measure latency and GPU board energy
def measure_query(model, tokenizer, query):
    messages = [
        {
            "role": "system",
            "content": "Answer the user's question clearly and concisely."
        },
        {"role": "user", "content": query}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(model.device)

    power_samples = []
    stop_sampling = threading.Event()

    def sample_power():
        while not stop_sampling.is_set():
            try:
                watts = pynvml.nvmlDeviceGetPowerUsage(
                    gpu_handle
                ) / 1000.0
                power_samples.append((time.perf_counter(), watts))
            except Exception:
                break
            stop_sampling.wait(POWER_SAMPLE_INTERVAL)

    # Start sampling immediately before inference
    sampler = None
    if POWER_AVAILABLE:
        sampler = threading.Thread(target=sample_power, daemon=True)
        sampler.start()

    synchronize_gpu()
    start = time.perf_counter()

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    synchronize_gpu()
    elapsed = time.perf_counter() - start

    if sampler is not None:
        stop_sampling.set()
        sampler.join(timeout=2)

    generated_tokens = output[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(
        generated_tokens, skip_special_tokens=True
    )

    # Integrate sampled power over time
    energy_j = None
    avg_power_w = None

    if len(power_samples) >= 2:
        energy_j = 0.0
        for i in range(1, len(power_samples)):
            t0, p0 = power_samples[i - 1]
            t1, p1 = power_samples[i]
            energy_j += (p0 + p1) / 2 * (t1 - t0)

        measured_duration = (
            power_samples[-1][0] - power_samples[0][0]
        )
        if measured_duration > 0:
            avg_power_w = energy_j / measured_duration
        else:
            energy_j = None

    return {
        "answer": answer,
        "latency_s": elapsed,
        "generated_tokens": len(generated_tokens),
        "tokens_per_s": len(generated_tokens) / elapsed if elapsed else None,
        "gpu_energy_j": energy_j,
        "avg_gpu_power_w": avg_power_w,
        "power_samples": len(power_samples),
    }


results = []

for model_name, model_id in MODELS.items():
    print(f"\n{'=' * 65}")
    print(f"Loading {model_name}: {model_id}")

    tokenizer = AutoTokenizer.from_pretrained(model_id)

    dtype = torch.float16 if torch.cuda.is_available() else torch.float32

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=dtype
    )
    if torch.cuda.is_available():
        model = model.to("cuda")
    model.eval()

    # Warm-up query
    _ = measure_query(
        model, tokenizer, "Hello! Please answer in one sentence."
    )
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    for i, query in enumerate(QUERIES, start=1):
        print(f"\nQuery {i}: {query}")

        measurement = measure_query(model, tokenizer, query)
        print("Answer:", measurement["answer"])
        print(f"Latency: {measurement['latency_s']:.3f} seconds")
        print(
            f"Generated tokens: {measurement['generated_tokens']}; "
            f"throughput: {measurement['tokens_per_s']:.2f} tokens/s"
        )

        if measurement["gpu_energy_j"] is not None:
            print(
                f"Estimated GPU-board energy: "
                f"{measurement['gpu_energy_j']:.3f} J"
            )
            print(
                f"Average sampled GPU power: "
                f"{measurement['avg_gpu_power_w']:.2f} W"
            )
        else:
            print("GPU energy unavailable for this measurement.")

        results.append({
            "model_name": model_name,
            "model_id": model_id,
            "query_number": i,
            "query": query,
            **measurement,
        })

    # Free memory before loading next model
    del model, tokenizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Save responses
df = pd.DataFrame(results)
df.to_csv("llm_energy_results.csv", index=False)

print("\nExperiment complete.")
display(df[[
    "model_name", "query_number", "latency_s",
    "generated_tokens", "tokens_per_s", "gpu_energy_j",
    "avg_gpu_power_w"
]])

print("\nSaved results to llm_energy_results.csv")


GPU power monitoring is available.

Loading small_1.5B: Qwen/Qwen2.5-1.5B-Instruct
The first load downloads the model and may take a few minutes.


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (15 outputs left out)

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


Query 1: What is the difference between RAM and storage?
Answer: RAM (Random Access Memory) is used for active data processing, while storage is for long-term data retention. RAM provides faster access to data but loses its contents when power is turned off, whereas storage holds data even without power and can be accessed more slowly.
Latency: 2.064 seconds
Generated tokens: 53; throughput: 25.68 tokens/s
Estimated GPU-board energy: 148.773 J
Average sampled GPU power: 72.34 W

Query 2: Summarize this paragraph in two sentences: Data centers consume electricity to power servers, networking equipment, and cooling systems. As AI workloads grow, operators are investigating more efficient hardware and software to reduce energy consumption.
Answer: Data centers use electricity for servers, networking, and cooling. To address growing AI workloads and reduce energy consumption, operators are exploring more efficient hardware and software solutions.
Latency: 1.318 seconds
Generated tokens: 3

,model_name,query_number,latency_s,generated_tokens,tokens_per_s,gpu_energy_j,avg_gpu_power_w
0,small_1.5B,1,1.395085,46,32.972913,85.732142,63.301189
1,small_1.5B,2,1.126834,38,33.722787,70.860192,64.232074
2,small_1.5B,3,3.857474,128,33.182338,244.352840,64.066091
3,small_1.5B,4,3.827026,128,33.446339,244.232876,64.063467
4,small_1.5B,5,3.872362,128,33.054759,246.474190,63.850227
5,large_3B,1,2.063934,53,25.679110,148.773195,72.342445
6,large_3B,2,1.318441,34,25.788033,94.373583,72.384585
7,large_3B,3,4.715766,122,25.870665,339.284297,72.723265
8,large_3B,4,2.726213,71,26.043453,197.168075,72.766862
9,large_3B,5,4.977892,128,25.713694,360.448042,72.576214



Saved results to llm_energy_results.csv


In [ ]:
# TODO: Grade generated responses